In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
Face = pd.read_csv('data/WM_Face_2bk_cleaned.csv')
Place = pd.read_csv('data/WM_Place_2bk_cleaned.csv')

In [ ]:
Place.iloc[:, 9:]

In [ ]:
Face.iloc[:, 9:]

In [ ]:
from scipy.stats import ttest_rel
import numpy as np

import matplotlib.pyplot as plt

# Align the dataframes based on the common Subject IDs
common_subjects = set(Face['Subject']).intersection(set(Place['Subject']))
Face_common = Face[Face['Subject'].isin(common_subjects)].sort_values(by='Subject')
Place_common = Place[Place['Subject'].isin(common_subjects)].sort_values(by='Subject')

# Perform paired t-tests for each column from 10 to the last column
columns_to_test = Face.columns[9:]
p_values = []
t_statistics = []

for col in columns_to_test:
    t_stat, p_val = ttest_rel(Face_common[col], Place_common[col])
    t_statistics.append(t_stat)
    p_values.append(p_val)

# Determine significance and direction
significance = np.array(p_values) < 0.05
direction = np.sign(t_statistics)

# Plot the results
plt.figure(figsize=(15, 5))
plt.bar(range(len(columns_to_test)), direction * significance, color=['red' if d > 0 else 'blue' for d in direction])
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xticks(range(len(columns_to_test)), columns_to_test, rotation=90, fontsize=8)
plt.ylabel('Significance Direction')
plt.title('Paired t-test Results (Red: Face > Place, Blue: Place > Face)')
plt.tight_layout()
plt.show()

In [ ]:
len(list(common_subjects))

In [ ]:
#performing fdr correction
from statsmodels.stats.multitest import multipletests
# Apply FDR correction
fdr_corrected = multipletests(p_values, alpha=0.05, method='fdr_bh')
# Extract the corrected p-values and significance
fdr_p_values = fdr_corrected[1]
significance_fdr = fdr_corrected[0]  # True if significant after FDR correction
# Plot the FDR-corrected results
plt.figure(figsize=(15, 5))
plt.bar(range(len(columns_to_test)), direction * significance_fdr, color=['red' if d > 0 else 'blue' for d in direction])
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xticks(range(len(columns_to_test)), columns_to_test, rotation=90, fontsize=8)
plt.ylabel('FDR Significance Direction')
plt.title('FDR Paired t-test Results (Red: Face > Place, Blue: Place > Face)')
plt.tight_layout()
plt.show()

In [ ]:
# Create a DataFrame for paired t-test results
t_test_results = pd.DataFrame({
    'ROI': columns_to_test,
    't_stat': t_statistics,
    'p_value_fdr': fdr_p_values,
    'significant_fdr': significance_fdr,
    'direction': ['face' if d > 0 else 'place' for d in direction]
})

# Display the DataFrame
t_test_results.head()

In [ ]:
t_test_results[t_test_results['ROI'] == 'R_FFC_ROI']

In [ ]:
t_test_results[(t_test_results['direction'] == 'face') & (t_test_results['significant_fdr'] == True)]

In [ ]:
t_test_results.to_csv('data/paired_t_test_results.csv', index=False)

## Combine paired sample t direction with face/place region types

In [ ]:
#read csvs with region types
Face_type = pd.read_csv('data/WM_Face_2bk_fdr_results_with_region_type.csv')
Place_type = pd.read_csv('data/WM_Place_2bk_fdr_results_with_region_type.csv')

In [ ]:
def manipulate_roi(roi):
    roi = roi.replace('_ROI', '')
    roi = roi.replace('.', '-')
    # if roi == 'L_10pp':
    #     roi = 'L_p10p'
    if roi.startswith('R'):
        return 'rh_' + roi
    elif roi.startswith('L'):
        return 'lh_' + roi
    return roi


In [ ]:
t_test_results['ROI'] = t_test_results['ROI'].apply(manipulate_roi)


In [ ]:
t_test_results.rename(columns={'ROI': 'label'}, inplace=True)
t_test_results.head()

In [ ]:
for row in t_test_results.iterrows():
    if not significance_fdr[row[0]]:
        t_test_results.at[row[0], 'direction'] = 'none'

In [ ]:
t_test_results

In [ ]:
t_test_results['direction'].value_counts()

In [ ]:
#check how many regions with direction face, their label start with 'lh_'
print('face left hemisphere:', t_test_results[(t_test_results['direction'] == 'face') & (t_test_results['label'].str.startswith('l'))].shape[0])
#check how many regions with direction face, their label start with 'rh_'
print('face right hemisphere:', t_test_results[(t_test_results['direction'] == 'face') & (t_test_results['label'].str.startswith('r'))].shape[0])
#check how many regions with direction place, their label start with 'lh_'
print('place left hemisphere:', t_test_results[(t_test_results['direction'] == 'place') & (t_test_results['label'].str.startswith('l'))].shape[0])
#check how many regions with direction place, their label start with 'rh_'
print('place right hemisphere:', t_test_results[(t_test_results['direction'] == 'place') & (t_test_results['label'].str.startswith('r'))].shape[0])
#check how many regions with direction none, their label start with 'lh_'
print('none left hemisphere:', t_test_results[(t_test_results['direction'] == 'none') & (t_test_results['label'].str.startswith('l'))].shape[0])
#check how many regions with direction none, their label start with 'rh_'
print('none right hemisphere:', t_test_results[(t_test_results['direction'] == 'none') & (t_test_results['label'].str.startswith('r'))].shape[0])

2025.4.14

try to use grok helping me write a 22 HCP region streams separation script, feeding the script with label column from each dataset and coming out the number of each stream's regions in a sheet, better to also list this regions' names out.

2025.4.15

`Debugged, functionally usable`

In [ ]:
def generate_stream_summary(df, condition_column=None, condition_value=None):
    """
    Generate a summary table of the number of regions in each of the 22 streams,
    optionally filtered by a condition.

    :param df: DataFrame containing the 'label' column with ROI names in format like 'rh_R_8Av'.
    :param condition_column: Optional column name to filter the DataFrame (e.g., 'direction').
    :param condition_value: Optional value to filter the DataFrame (e.g., 'face').
    :return: DataFrame with 'stream' and 'count' columns.
    """
    # Define the 22 streams and their areas based on HCP MMP1.0 atlas
    streams = [
        ("Primary Visual Cortex", ['V1']),
        ("Early Visual Cortex", ['V2', 'V3', 'V4']),
        ("Dorsal Stream Visual Cortex", ['V3A', 'V3B', 'V6', 'V6A', 'V7', 'IPS1']),
        ("Ventral Stream Visual Cortex", ['PIT', 'FFC', 'VMV1', 'VMV2', 'VMV3', 'V8', 'VVC']),
        ("MT+ Complex and Its Neighbors", ['MT', 'MST', 'FST', 'V4t', 'LO1', 'LO2', 'LO3', 'PH', 'V3CD']),
        ("Early Somatosensory and Motor Cortex", ['3a', '3b', '1', '2', '4']),
        ("Sensori-motor Associated Paracentral Lobular and Mid Cingulate Cortex", ['5m', '5L', '5mv', '24dd', '24dv', '6mp', '6ma', 'SCEF']),
        ("Premotor Cortex", ['6a', '6d', '6v', 'FEF', '55b', '6r', 'PEF']),
        ("Posterior Opercular Cortex", ['FOP1', '43', 'OP4', 'OP1', 'PFcm', 'OP2-3']),
        ("Early Auditory Cortex", ['A1', 'MBelt', 'LBelt', 'PBelt', 'RI']),
        ("Association Auditory Cortex", ['STSda', 'STSdp','STSva', 'STSvp', 'STGa', 'A4', 'A5', 'TA2']),
        ("Insular and Frontal Opercular Cortex", ['52', 'PI', 'Ig', 'Pir', 'AVI', 'AAIC', 'MI', 'PoI1', 'PoI2','FOP2', 'FOP3', 'FOP4', 'FOP5']),
        ("Medial Temporal Cortex", ['H', 'PreS', 'PHA1', 'PHA2', 'PHA3', 'PeEc', 'EC']),
        ("Lateral Temporal Cortex", ['TE1a', 'TE1m', 'TE1p', 'TE2a', 'TE2p', 'TGv', 'TGd', 'TF', 'PHT']),
        ("Temporal-Parietal-Occipital Junction", ['TPOJ1', 'TPOJ2', 'TPOJ3', 'PSL', 'STV']),
        ("Superior Parietal and IPS Cortex", ['LIPd', 'LIPv', 'VIP', 'AIP', 'MIP', '7PC', '7AL', '7Am', '7PL', '7Pm']),
        ("Inferior Parietal Cortex", ['PF', 'PFt', 'PFm', 'PGp', 'PGi', 'PGs', 'PFop', 'IP0', 'IP1', 'IP2']),
        ("Posterior Cingulate Cortex", ['DVT', 'ProS', 'POS1', 'POS2', 'RSC', 'v23ab', 'd23ab', '31pv', '31pd', '31a', '23d', '23c', 'PCV', '7m']),
        ("Anterior Cingulate and Medial Prefrontal Cortex", ['33pr', 'p24pr', 'a24pr', 'p24', 'a24', 'p32pr', 'a32pr', 'd32', 'p32', 's32', '8BM', '9m', '10v', '10r', '25']),
        ("Orbital and Polar Frontal Cortex", ['47s', '47m', 'a47r', '11l' ,'13l', 'a10p', 'p10p', '10pp', '10d', 'OFC', 'pOFC']),
        ("Inferior Frontal Cortex", ['44', '45', 'IFJa', 'IFJp', '47l', 'p47r', 'IFSa', 'IFSp']),
        ("Dorsolateral Prefrontal Cortex", ['8C', '8Av', 'i6-8', 's6-8', 'SFL', '8BL', '9p', '9a', '8Ad', 'p9-46v', 'a9-46v', '46', '9-46d']),
    ]

    # Create a mapping from area names to streams
    area_to_stream = {}
    for stream_name, areas in streams:
        for area in areas:
            area_to_stream[area] = stream_name

    # Create a mapping from stream names to their sizes (number of areas per stream)
    size_map = {stream_name: len(areas) for stream_name, areas in streams}

    # Function to extract area name from label
    def extract_area(label):
        parts = label.split('_')
        if len(parts) >= 3:
            return '_'.join(parts[2:])
        return None

    # Add 'area' column to DataFrame by parsing 'label'
    df = df.copy()  # Avoid modifying the original DataFrame
    df['area'] = df['label'].apply(extract_area)

    # Map areas to their streams
    df['stream'] = df['area'].map(area_to_stream)

    # Map areas to their hemisphere
    df['hemisphere'] = df['label'].apply(lambda x: 'lh' if x.startswith('lh') else 'rh')

    # Apply filtering if a condition is provided
    if condition_column and condition_value:
        filtered_df = df[df[condition_column] == condition_value].copy()
    else:
        filtered_df = df.copy()

    # Compute counts before reindexing
    # Total count per stream
    total_counts = filtered_df.groupby('stream').size().to_dict()

    # Left hemisphere count per stream
    lh_counts = filtered_df[filtered_df['hemisphere'] == 'lh'].groupby('stream').size().to_dict()

    # Right hemisphere count per stream
    rh_counts = filtered_df[filtered_df['hemisphere'] == 'rh'].groupby('stream').size().to_dict()

    # Create summary DataFrame with all streams
    all_streams = [stream[0] for stream in streams]
    summary = pd.DataFrame({'stream': all_streams})

    # Add counts, defaulting to 0 if the stream is not present
    summary['count'] = summary['stream'].map(total_counts).fillna(0).astype(int)
    summary['lh_count'] = summary['stream'].map(lh_counts).fillna(0).astype(int)
    summary['rh_count'] = summary['stream'].map(rh_counts).fillna(0).astype(int)

    # Add Total_size and Per_Hemisphere_size columns (fixed values based on streams)
    summary['Per_Hemisphere_size'] = summary['stream'].map(size_map)
    summary['Total_size'] = summary['Per_Hemisphere_size'] * 2  # Total size is per-hemisphere size times 2

    return summary

In [ ]:
summary_face = generate_stream_summary(t_test_results, condition_column='direction', condition_value='face')
summary_place = generate_stream_summary(t_test_results, condition_column='direction', condition_value='place')

In [ ]:
summary_place

In [ ]:
t_test_results

In [ ]:
Face_type = Face_type.merge(t_test_results[['label', 'direction']], on='label', how='left')
Place_type = Place_type.merge(t_test_results[['label', 'direction']], on='label', how='left')

`save the file`

In [ ]:
Face_type.to_csv('data/Face_results_with_t_direction.csv', index=False)
Place_type.to_csv('data/Place_results_with_t_direction.csv', index=False)

`check each task's regions' directions`

In [ ]:
Face_type['direction'].value_counts()

In [ ]:
Face_type

In [ ]:
generate_stream_summary(Face_type, condition_column='direction', condition_value='place')

In [ ]:
Place_type['direction'].value_counts()

In [ ]:
generate_stream_summary(Place_type, condition_column='direction', condition_value='face')

In [ ]:
#none regions in face task gam significant regions
generate_stream_summary(Face_type, condition_column='direction', condition_value='none')

In [ ]:
Face_type[Face_type['direction'] == 'none']

In [ ]:
#none regions in place task gam significant regions
generate_stream_summary(Place_type, condition_column='direction', condition_value='none')

In [ ]:
Place_type[Place_type['direction'] == 'none']

# One sample t-test results

In [ ]:
from scipy.stats import ttest_1samp

# Perform one-sample t-tests for Face dataset
face_t_statistics = []
face_p_values = []

for col in columns_to_test:
    t_stat, p_val = ttest_1samp(Face[col], popmean=0, nan_policy='omit')
    face_t_statistics.append(t_stat)
    face_p_values.append(p_val)

# Perform one-sample t-tests for Place dataset
place_t_statistics = []
place_p_values = []

for col in columns_to_test:
    t_stat, p_val = ttest_1samp(Place[col], popmean=0, nan_policy='omit')
    place_t_statistics.append(t_stat)
    place_p_values.append(p_val)

# Create DataFrames for the results
face_t_test_results = pd.DataFrame({
    'ROI': columns_to_test,
    't_stat': face_t_statistics,
    'p_value': face_p_values
})

place_t_test_results = pd.DataFrame({
    'ROI': columns_to_test,
    't_stat': place_t_statistics,
    'p_value': place_p_values
})

# Display the first few rows of the results
print("Face Dataset One-Sample T-Test Results:")
print(face_t_test_results.head())

print("\nPlace Dataset One-Sample T-Test Results:")
print(place_t_test_results.head())

In [ ]:
face_fdr_corrected = multipletests(face_p_values, alpha=0.05, method='fdr_bh')
face_fdr_p_values = face_fdr_corrected[1]
face_significance_fdr = face_fdr_corrected[0]

# Perform FDR correction for Place dataset
place_fdr_corrected = multipletests(place_p_values, alpha=0.05, method='fdr_bh')
place_fdr_p_values = place_fdr_corrected[1]
place_significance_fdr = place_fdr_corrected[0]

# Add FDR-corrected results to the DataFrames
face_t_test_results['p_value_fdr'] = face_fdr_p_values
face_t_test_results['significant_fdr'] = face_significance_fdr

place_t_test_results['p_value_fdr'] = place_fdr_p_values
place_t_test_results['significant_fdr'] = place_significance_fdr

# Display the first few rows of the updated results
print("Face Dataset One-Sample T-Test Results with FDR Correction:")
print(face_t_test_results.head())

print("\nPlace Dataset One-Sample T-Test Results with FDR Correction:")
print(place_t_test_results.head())

In [ ]:
face_t_test_results[face_t_test_results['significant_fdr'] == False]

In [ ]:
place_t_test_results[place_t_test_results['significant_fdr'] == False]

In [ ]:
# Regions with significant_fdr in both Face and Place t-test results
common_significant_regions_results = face_t_test_results[
    (face_t_test_results['significant_fdr'] == True) & 
    (place_t_test_results['significant_fdr'] == True)
]


In [ ]:
common_significant_regions_results.shape